# ============================================================
# STEP 7B — URL-TRANSFORMER ATTENTION VISUALIZATION (TUMC)
# ============================================================
# Loads the trained character-level URL-Transformer (Step 7) and
# visualizes which characters the self-attention focuses on for
# example URLs. This is the explainability counterpart to the
# transformer baseline: it shows whether attention concentrates
# on the Turkish brand/action tokens (e.g. "garanti", "giris")
# that mark phishing — evidence the model learned interpretable
# structure from raw characters, not noise.
#
# METHOD:
#   - Register forward hooks on each encoder layer's self_attn,
#     forcing need_weights=True (PyTorch's fast path is disabled
#     so weights are actually returned).
#   - Per-character salience = attention RECEIVED, i.e. for the
#     chosen layer, average over heads then mean over query rows
#     (column-mean of the seq x seq map), masked to non-pad chars,
#     normalized to [0,1].
#   - Plot: (1) a colored character strip (heatmap over the URL),
#     (2) a bar profile of salience per character.
#
# OUTPUTS (per example URL):
#   step7b_attention_<label>.png
#   step7b_attention_summary.png  (grid of all examples)
# ============================================================

In [ ]:


import os, math, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
MODEL_IN = os.path.join(DATA_DIR, "url_transformer_best.pt")
OUT_DIR  = os.path.join(DATA_DIR, "attention_viz")
os.makedirs(OUT_DIR, exist_ok=True)

# ── MUST MATCH STEP 7 EXACTLY ────────────────────────────────
MAX_LEN=200; D_MODEL=64; N_HEADS=4; N_LAYERS=3; D_FF=128; DROPOUT=0.1
PAD_IDX=0; UNK_IDX=1
CHARS = ("abcdefghijklmnopqrstuvwxyz" "0123456789"
         ".-_/:?=&%@+~#!$*()[]{}|\\;,<>\"' " "çğıöşüâî")
char2idx = {c: i+2 for i,c in enumerate(CHARS)}
VOCAB_SIZE = len(char2idx) + 2

# Map class index -> name (match Step 7 label encoding; adjust if needed)
CLASS_NAMES = ["benign","malware","other_malicious","phishing","scam"]  # alpha order of class_enc
# If our class_enc uses a different order, set it here to match training.

def encode_url(url, max_len=MAX_LEN):
    url = str(url).lower()[:max_len]
    ids = [char2idx.get(c, UNK_IDX) for c in url]
    ids += [PAD_IDX]*(max_len-len(ids))
    return ids, list(url)   # also return the visible chars

# ── MODEL (identical to Step 7) ──────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()*(-math.log(10000)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self,x): return x + self.pe[:, :x.size(1)]

class URLTransformer(nn.Module):
    def __init__(self, vocab_size, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, D_MODEL, padding_idx=PAD_IDX)
        self.pos   = PositionalEncoding(D_MODEL)
        enc = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT,
                                         activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, N_LAYERS)
        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL*2, D_MODEL), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(D_MODEL, n_classes))
    def forward(self, x):
        mask = (x == PAD_IDX)
        h = self.embed(x)*math.sqrt(D_MODEL); h = self.pos(h)
        h = self.encoder(h, src_key_padding_mask=mask)
        hm = h.masked_fill(mask.unsqueeze(-1),0).sum(1)/(~mask).sum(1,keepdim=True).clamp(min=1)
        hx = h.masked_fill(mask.unsqueeze(-1),-1e9).max(1).values
        return self.head(self.dropout(torch.cat([hm,hx],1)))

# ── LOAD TRAINED WEIGHTS ─────────────────────────────────────
print("="*70); print("STEP 7B — ATTENTION VISUALIZATION"); print("="*70)
n_classes = len(CLASS_NAMES)
model = URLTransformer(VOCAB_SIZE, n_classes).to(DEVICE)
state = torch.load(MODEL_IN, map_location=DEVICE)
state = state.get("model_state", state) if isinstance(state, dict) and "model_state" in state else state
model.load_state_dict(state)
model.eval()
print(f"  loaded {MODEL_IN}")

# ── ATTENTION HOOKS (force weights; disable fast path) ───────
_attn = []
def _hook(mod, inp, out):
    if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
        _attn.append(out[1].detach().cpu())   # (B, heads, S, S)
for layer in model.encoder.layers:
    # disable the fused fast-path so python self_attn runs and returns weights
    layer.self_attn.register_forward_hook(_hook)
    _orig = layer.self_attn.forward
    def _patched(*a, _o=_orig, **kw):
        kw["need_weights"]=True; kw["average_attn_weights"]=False
        return _o(*a, **kw)
    layer.self_attn.forward = _patched
# also turn off encoder fast path explicitly
model.encoder.enable_nested_tensor = False

def char_salience(url, layer="last"):
    """Return (chars, salience[0..1], predicted_class, prob)."""
    ids, chars = encode_url(url)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    _attn.clear()
    with torch.no_grad():
        logits = model(x)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())
    L = len(chars)
    if not _attn:
        return chars, np.zeros(L), pred, float(probs[pred])
    maps = torch.stack(_attn, 0)               # (layers, B, heads, S, S)
    sel = maps[-1] if layer == "last" else maps.mean(0)
    sel = sel[0].mean(0)                        # average heads -> (S, S)
    received = sel[:, :L].mean(0).numpy()       # column-mean over queries -> (L,)
    if received.max() > received.min():
        received = (received - received.min())/(received.max()-received.min())
    return chars, received, pred, float(probs[pred])

# ── PLOTTING ─────────────────────────────────────────────────
CMAP = LinearSegmentedColormap.from_list("sal", ["#f7f7f7","#fdae61","#d73027"])

def plot_attention(url, fname, layer="last"):
    chars, sal, pred, prob = char_salience(url, layer)
    L = len(chars)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(min(0.22*L+1.5, 16), 3.2),
                                   gridspec_kw={"height_ratios":[1,2]})
    # strip of colored characters
    ax1.set_xlim(0, L); ax1.set_ylim(0, 1); ax1.axis("off")
    for i, c in enumerate(chars):
        ax1.add_patch(plt.Rectangle((i,0),1,1, color=CMAP(sal[i])))
        ax1.text(i+0.5, 0.5, c, ha="center", va="center",
                 fontsize=11, family="monospace",
                 color="white" if sal[i]>0.6 else "black")
    ax1.set_title(f"URL-Transformer attention  →  predicted: "
                  f"{CLASS_NAMES[pred]} ({prob:.2f})",
                  fontsize=11, pad=6)
    # salience bars
    ax2.bar(range(L), sal, color=[CMAP(s) for s in sal], width=0.9)
    ax2.set_xlim(-0.5, L-0.5); ax2.set_ylim(0,1.05)
    ax2.set_xticks(range(L))
    ax2.set_xticklabels(chars, fontsize=8, family="monospace")
    ax2.set_ylabel("attention\n(normalized)", fontsize=9)
    ax2.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, fname)
    plt.savefig(out, dpi=160, bbox_inches="tight"); plt.close()
    print(f"  saved {out}  (pred={CLASS_NAMES[pred]} {prob:.2f})")
    # print top-attended substring for the caption
    order = np.argsort(-sal)[:8]
    top = "".join(chars[i] for i in sorted(order))
    print(f"     top-attended chars: '{top}'")
    return chars, sal, pred, prob

# ── EXAMPLE URLs (edit to taste) ─────────────────────────────
EXAMPLES = [
    ("https://www.ktun.edu.tr/",                       "Konya Technical University"),
    ("https://giris.turkiye.gov.tr/Giris/gir", "E-devlet Official Website"),
    ("http://trendyol-kampanya-hediye.tk",      "trendyol_scam"),
    ("https://www.hurriyet.com.tr/gundem",      "benign_news"),
]

print("\n[Generating per-URL attention figures]")
results = []
for url, tag in EXAMPLES:
    r = plot_attention(url, f"step7b_attention_{tag}.png", layer="last")
    results.append((url, *r))

# ── SUMMARY GRID ─────────────────────────────────────────────
n = len(results)
fig, axes = plt.subplots(n, 1, figsize=(14, 1.5*n))
if n == 1: axes=[axes]
for ax,(url,chars,sal,pred,prob) in zip(axes, results):
    L=len(chars); ax.set_xlim(0,L); ax.set_ylim(0,1); ax.axis("off")
    for i,c in enumerate(chars):
        ax.add_patch(plt.Rectangle((i,0),1,1,color=CMAP(sal[i])))
        ax.text(i+0.5,0.5,c,ha="center",va="center",fontsize=9,
                family="monospace",color="white" if sal[i]>0.6 else "black")
    ax.text(L+0.5,0.5,f"{CLASS_NAMES[pred]} ({prob:.2f})",
            va="center",fontsize=9)
fig.suptitle("URL-Transformer character attention (last layer, normalized)",
             fontsize=12, y=1.02)
plt.tight_layout()
grid_out=os.path.join(OUT_DIR,"step7b_attention_summary.png")
plt.savefig(grid_out, dpi=160, bbox_inches="tight"); plt.close()
print(f"\n  saved summary grid: {grid_out}")

print("\n" + "="*70)
print("INTERPRETATION (for the manuscript)")
print("="*70)
print("  If attention concentrates on Turkish brand/action tokens")
print("  ('ktun','giris','e-devlet','dogrula') on phishing URLs and")
print("  spreads diffusely on benign URLs, the figure demonstrates the")
print("  transformer learned interpretable Turkish phishing structure")
print("  from raw characters — complementing the SHAP/IG analysis of the")
print("  engineered-feature models. Note the transformer still trails the")
print("  gradient-boosted trees on macro-F1 (0.76 vs 0.90), so attention")
print("  is presented as an explainability artifact, not the deployed model.")

STEP 7B — ATTENTION VISUALIZATION
  loaded /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/url_transformer_best.pt

[Generating per-URL attention figures]
  saved /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/attention_viz/step7b_attention_garanti_giris.png  (pred=benign 1.00)
     top-attended chars: 'hwwwkut/'
  saved /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/attention_viz/step7b_attention_edevlet_dogrula.png  (pred=benign 0.98)
     top-attended chars: 'h//iiovi'
  saved /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/attention_viz/step7b_attention_trendyol_scam.png  (pred=phishing 0.97)
     top-attended chars: 'en-kydyk'
  saved /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/attention_viz/step7b_attention_benign_news.png  (pred=benign 0.99)
     top-attended chars: 'hwww/udm'

  saved summary grid: /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/attention_viz/step7b_attention_s